# Module 6: Clinic APIs (Offline Practice)

**Concepts:** HTTP verbs · query params · headers · timeouts · fake offline client

> **Safety note:** This is a **programming** exercise. It is **not** medical advice, not a diagnosis, and **not** for real patients or real medicines. We use invented drug names (for example **Alercin**) and school biology/chemistry only.

We **do not** call the real internet. We use tiny fake responses so you can learn offline (no `requests` install needed).

**Words**

| Word | Meaning |
|------|---------|
| **GET** | read data |
| **POST** | send new data |
| **timeout** | give up if the server is too slow |

## 1. Status codes (mental model)

**What you should see:** ok / client / server labels.

In [ ]:
def classify_status(code):
    if 200 <= code <= 299:
        return "ok"
    if 400 <= code <= 499:
        return "client"
    if 500 <= code <= 599:
        return "server"
    return "other"


for c in (200, 404, 503):
    print(c, classify_status(c))

## 2. Query parameters

**What you should see:** a URL with `drug=Alercin`.

In [ ]:
from urllib.parse import urlencode

base = "https://clinic.example/api/formulary"
query = urlencode({"drug": "Alercin", "form": "tablet"})
print(f"{base}?{query}")

## 3. Headers — never commit real tokens

**What you should see:** a Bearer header built from a fake practice token.

In [ ]:
import os

os.environ["CLINIC_TOKEN"] = "practice-token-not-real"


def auth_headers(token_env="CLINIC_TOKEN"):
    token = os.environ.get(token_env, "")
    return {"Authorization": f"Bearer {token}"}


print(auth_headers())

## 4. Always set timeouts

**What you should see:** the timeout tuple printed.

In [ ]:
DEFAULT_TIMEOUT = (3.0, 10.0)  # connect, read seconds
print("use timeout", DEFAULT_TIMEOUT)

## 5. Fake GET — formulary card for Alercin

No internet and no `requests` library needed. We build a tiny fake response object.

**What you should see:** JSON-like dict with invented drug info.

In [ ]:
class FakeResponse:
    def __init__(self, status_code, data):
        self.status_code = status_code
        self._data = data

    def json(self):
        return self._data

    def raise_for_status(self):
        if self.status_code >= 400:
            raise Exception(f"HTTP {self.status_code}")


def fake_get(url, headers=None, timeout=None):
    print("GET", url, "timeout=", timeout)
    return FakeResponse(
        200,
        {"name": "Alercin", "mg_per_kg": 10, "max_mg": 1000, "form": "tablet"},
    )


r = fake_get(
    "https://clinic.example/api/formulary/Alercin",
    headers={"Authorization": "Bearer practice"},
    timeout=5,
)
r.raise_for_status()
print(r.json())

## 6. Fake POST — send a vital measurement

**What you should see:** `{"stored": True}`.

In [ ]:
def fake_post(url, json=None, timeout=None):
    print("POST", url, "body=", json)
    return FakeResponse(201, {"stored": True})


r = fake_post(
    "https://clinic.example/api/vitals",
    json={"patient": "Ana", "temp_c": 37.8},
    timeout=5,
)
r.raise_for_status()
print(r.json())

## 7. Capstone — `FormularyClient` class

The client calls a injectable `get_fn` so tests stay offline.

**What you should see:** drug card from the client helper.

In [ ]:
class FormularyClient:
    def __init__(self, base, token, timeout=10.0, get_fn=None):
        self.base = base.rstrip("/")
        self.token = token
        self.timeout = timeout
        self.get_fn = get_fn or fake_get

    def headers(self):
        return {"Authorization": f"Bearer {self.token}"}

    def get_drug(self, name):
        url = f"{self.base}/formulary/{name}"
        r = self.get_fn(url, headers=self.headers(), timeout=self.timeout)
        r.raise_for_status()
        return r.json()


client = FormularyClient("https://clinic.example/api", "practice")
print(client.get_drug("Alercin"))

---
# Exercises

**Ex 1** — Function building Authorization Bearer header from a token string.

In [ ]:
# TODO

**Ex 2** — FakeResponse with status 503; call `raise_for_status` in try/except.

In [ ]:
# TODO

**Ex 3** — Add `FormularyClient.vitals_url()` returning `base + "/vitals"`.

In [ ]:
# TODO

---
# Solutions

In [ ]:
def bearer(token):
    return {"Authorization": f"Bearer {token}"}


print(bearer("practice"))

In [ ]:
class FakeResponse:
    def __init__(self, status_code, data=None):
        self.status_code = status_code
        self._data = data or {}

    def raise_for_status(self):
        if self.status_code >= 400:
            raise Exception(f"HTTP {self.status_code}")


try:
    FakeResponse(503).raise_for_status()
except Exception as e:
    print("handled", e)

In [ ]:
class FormularyClient:
    def __init__(self, base, token):
        self.base = base.rstrip("/")
        self.token = token

    def vitals_url(self):
        return f"{self.base}/vitals"


print(FormularyClient("https://clinic.example/api", "t").vitals_url())

**Next:** Module 7 — type hints and tests for dose, DNA, Punnett.